In [1]:
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
from pathlib import Path
import librosa
from huggingface_hub import login
import torch
from transformers import AutoModel
from tqdm import tqdm
from transformers import AutoFeatureExtractor

# import opensmile

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, StratifiedGroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

## Data Preparation

In [2]:
dementia_root_path = Path("D:/NUS/Semester 2/Innovation Challenge/Code/data/dementia")
nodementia_root_path = Path("D:/NUS/Semester 2/Innovation Challenge/Code/data/nodementia")
data = []

# Use rglob to recursively find all audio files (e.g., .wav; change to .mp3 if needed)
for file_path in dementia_root_path.rglob('*.wav'):
    # Parse path structure
    # file_path.parent is the subject folder (e.g., patient_01)
    # file_path.parent.parent is the category folder (dementia or non-dementia)
    subject_id = file_path.parent.name
    label_name = file_path.parent.parent.name

    # Convert label to numerical value for model training (dementia: 1, non-dementia: 0)
    label = 1 if label_name == 'dementia' else 0

    data.append({
        'subject_id': subject_id,
        'label': label,
        'file_path': str(file_path)
    })

for file_path in nodementia_root_path.rglob('*.wav'):
    # Parse path structure
    # file_path.parent is the subject folder (e.g., patient_01)
    # file_path.parent.parent is the category folder (dementia or non-dementia)
    subject_id = file_path.parent.name
    label_name = file_path.parent.parent.name

    # Convert label to numerical value for model training (dementia: 1, non-dementia: 0)
    label = 1 if label_name == 'dementia' else 0

    data.append({
        'subject_id': subject_id,
        'label': label,
        'file_path': str(file_path)
    })

# Convert the prepared list to a DataFrame
df = pd.DataFrame(data)

# View the first few rows of data
print(df.head())
df

         subject_id  label                                          file_path
0       Abe Burrows      1  D:\NUS\Semester 2\Innovation Challenge\Code\da...
1  Aileen Hernandez      1  D:\NUS\Semester 2\Innovation Challenge\Code\da...
2  Aileen Hernandez      1  D:\NUS\Semester 2\Innovation Challenge\Code\da...
3  Aileen Hernandez      1  D:\NUS\Semester 2\Innovation Challenge\Code\da...
4       Alan Ramsey      1  D:\NUS\Semester 2\Innovation Challenge\Code\da...


,subject_id,label,file_path
0,Abe Burrows,1,D:\NUS\Semester 2\Innovation Challenge\Code\da...
1,Aileen Hernandez,1,D:\NUS\Semester 2\Innovation Challenge\Code\da...
2,Aileen Hernandez,1,D:\NUS\Semester 2\Innovation Challenge\Code\da...
3,Aileen Hernandez,1,D:\NUS\Semester 2\Innovation Challenge\Code\da...
4,Alan Ramsey,1,D:\NUS\Semester 2\Innovation Challenge\Code\da...
...,...,...,...
450,William Shatner,0,D:\NUS\Semester 2\Innovation Challenge\Code\da...
451,William Shatner,0,D:\NUS\Semester 2\Innovation Challenge\Code\da...
452,Yoko Ono,0,D:\NUS\Semester 2\Innovation Challenge\Code\da...
453,Yoko Ono,0,D:\NUS\Semester 2\Innovation Challenge\Code\da...


## Model

In [3]:
load_dotenv()

HF_TOKEN = os.getenv('HF_TOKEN')
login(token = HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
# ================= 1. Environment Setup and Model Loading ====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "google/hear-pytorch"

# Google HeAR typically requires a dedicated Feature Extractor to convert audio into a format acceptable by the model.
try:
    processor = AutoFeatureExtractor.from_pretrained(model_id, trust_remote_code=True)
except Exception as e:
    print(f"Failed to load processor: {e}")
    # Some custom models may not have a standard processor; if so, we might need to manually process into spectrograms.
    processor = None

model = AutoModel.from_pretrained(model_id, trust_remote_code=True).to(device)
model.eval()

Failed to load processor: Can't load feature extractor for 'google/hear-pytorch'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'google/hear-pytorch' is the correct path to a directory containing a preprocessor_config.json file


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "c:\Users\horat\miniconda3\envs\data\Lib\site-packages\huggingface_hub\utils\_http.py", line 657, in hf_raise_for_status
    response.raise_for_status()
  File "c:\Users\horat\miniconda3\envs\data\Lib\site-packages\httpx\_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/google/hear-pytorch/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\horat\miniconda3\envs\data\Lib\threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "c:\Users\horat\miniconda3\envs\data\Lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  Fi

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(1, 1024, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-23): 24 x ViTLayer(
        (attention): ViTAttention(
          (attention): ViTSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=1024, out_features=4096, bias=True)
          (intermediate_act_fn): FastGELUActivation()
        )
        (output): ViTOutput

In [5]:
def load_and_chunk_audio(file_path, target_sr=16000, chunk_duration=2.0):
    """ Loads an audio file and chunks it into segments of a specified duration. """
    try:
        # Load audio, convert to mono and 16kHz
        y, sr = librosa.load(file_path, sr=target_sr, mono=True)

        # Calculate the number of data points needed for each chunk (16000 * 2 = 32000)
        chunk_samples = int(target_sr * chunk_duration)
        chunks = []

        # Use a loop to perform chunking
        for i in range(0, len(y), chunk_samples):
            chunk = y[i:i + chunk_samples]
            # Handle tail segments less than 2 seconds: choose to discard or pad with 0 (silence)
            # Here, we strictly require 2s chunks and discard segments that are too short.
            if len(chunk) == chunk_samples:
                chunks.append(chunk)

        return chunks
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return []

In [6]:
# Enable tqdm progress bar support for Pandas
tqdm.pandas(desc="Processing audio files")

# Apply the load_and_chunk_audio function to each row in the 'file_path' column
# And store the returned list of chunks into a new column 'audio_chunks'
df['audio_chunks'] = df['file_path'].progress_apply(load_and_chunk_audio)

# Since some audio files might fail to load, or their total duration is less than 2 seconds resulting in an empty list []
# We need to filter out these invalid rows to create a clean DataFrame
df_clean = df[df['audio_chunks'].map(len) > 0].copy()

# Reset the index to keep the data tidy
df_clean.reset_index(drop=True, inplace=True)

print(f"Original number of rows: {len(df)}")
print(f"Number of valid rows with successfully chunked audio: {len(df_clean)}")

Processing audio files: 100%|██████████| 455/455 [00:32<00:00, 14.08it/s]

Original number of rows: 455
Number of valid rows with successfully chunked audio: 455


In [7]:
df_clean

,subject_id,label,file_path,audio_chunks
0,Abe Burrows,1,D:\NUS\Semester 2\Innovation Challenge\Code\da...,"[[0.0036099176, 0.006115362, 0.0066214297, 0.0..."
1,Aileen Hernandez,1,D:\NUS\Semester 2\Innovation Challenge\Code\da...,"[[-0.0013588178, -0.0015186558, -0.00079729105..."
2,Aileen Hernandez,1,D:\NUS\Semester 2\Innovation Challenge\Code\da...,"[[0.0030658355, 5.0986942e-05, -0.0069100135, ..."
3,Aileen Hernandez,1,D:\NUS\Semester 2\Innovation Challenge\Code\da...,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
4,Alan Ramsey,1,D:\NUS\Semester 2\Innovation Challenge\Code\da...,"[[-2.5072616e-06, 1.7858645e-05, -3.9076014e-0..."
...,...,...,...,...
450,William Shatner,0,D:\NUS\Semester 2\Innovation Challenge\Code\da...,"[[-0.00064473134, 0.003295409, 0.0066730413, 0..."
451,William Shatner,0,D:\NUS\Semester 2\Innovation Challenge\Code\da...,"[[-0.0027777168, 0.0061449734, 0.009427748, -0..."
452,Yoko Ono,0,D:\NUS\Semester 2\Innovation Challenge\Code\da...,"[[-0.02066085, -0.03636156, -0.031466722, -0.0..."
453,Yoko Ono,0,D:\NUS\Semester 2\Innovation Challenge\Code\da...,"[[0.013117724, 0.019692227, 0.012761049, 0.010..."


## Version 1

In [23]:
# def extract_hear_embeddings(chunks, model, processor, device):
#     if not chunks or len(chunks) == 0:
#         return None

#     embeddings = []
#     model.eval()

#     target_frames = 192  # 時間軸
#     target_mels = 128    # 頻率軸

#     with torch.no_grad():
#         for chunk in chunks:
#             S = librosa.feature.melspectrogram(
#                 y=chunk, sr=16000, n_mels=128, hop_length=160
#             )
#             S_db = librosa.power_to_db(S, ref=1.0)

#             # 調整時間軸到 192
#             if S_db.shape[1] > target_frames:
#                 S_db = S_db[:, :target_frames]
#             else:
#                 pad_width = target_frames - S_db.shape[1]
#                 S_db = np.pad(S_db, ((0, 0), (0, pad_width)), mode='constant')

#             # Transpose: (128, 192) → (192, 128)
#             S_db_transposed = S_db.T

#             # unsqueeze → (1, 1, 192, 128)
#             input_tensor = torch.tensor(
#                 S_db_transposed, dtype=torch.float32
#             ).unsqueeze(0).unsqueeze(0).to(device)

#             try:
#                 output = model(input_tensor)

#                 if hasattr(output, 'pooler_output') and output.pooler_output is not None:
#                     emb = output.pooler_output.cpu().numpy().flatten()
#                 elif hasattr(output, 'last_hidden_state'):
#                     emb = output.last_hidden_state.mean(dim=1).cpu().numpy().flatten()
#                 else:
#                     emb = output[0].mean(dim=1).cpu().numpy().flatten()

#                 embeddings.append(emb)

#             except Exception as e:
#                 print(f"Model Inference Error: {e}")
#                 continue

#     return np.mean(embeddings, axis=0) if embeddings else None

In [27]:
# embeddings_list = []

# # Use tqdm to display the progress bar
# print(f"Starting embedding extraction on {len(df_clean)} rows...")

# # Switch model to eval mode and specify device
# model.eval()
# model.to(device)

# for idx, row in tqdm(df_clean.iterrows(), total=len(df_clean), desc="Extracting"):
#     chunks = row['audio_chunks']

#     # Call our previously defined function
#     # Note: Ensure extract_hear_embeddings is defined within scope
#     try:
#         embedding = extract_hear_embeddings(chunks, model, processor, device)

#         if embedding is not None:
#             embeddings_list.append(embedding)
#         else:
#             # If chunks is empty, append a zero vector to maintain consistent dimensions (assuming embedding dimension is 1024)
#             # In practice, it is recommended to first confirm the model's embedding size
#             embeddings_list.append(None)

#     except Exception as e:
#         print(f"Error processing row {idx} (File: {row['file_path']}): {e}")
#         embeddings_list.append(None)

# # Add results to a new column
# df_clean['embedding'] = embeddings_list

# # Remove invalid rows where embedding could not be extracted
# df_result = df_clean.dropna(subset=['embedding']).copy()

Starting embedding extraction on 455 rows...


Extracting:   8%|▊         | 36/455 [00:00<00:01, 352.24it/s]

Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneTyp

Extracting:  24%|██▎       | 107/455 [00:00<00:01, 339.67it/s]

Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneTyp

...

Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneTyp

Extracting:  83%|████████▎ | 379/455 [00:01<00:00, 338.24it/s]

Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneTyp

Extracting: 100%|██████████| 455/455 [00:01<00:00, 331.85it/s]

Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneType' object is not callable
Model Inference Error: 'NoneTyp

## Version 2

In [8]:
# 下載 Google 官方 HeAR 儲存庫
# !git clone https://github.com/google-health/hear.git

# 將儲存庫路徑加入系統環境以便載入
import sys
sys.path.append('/content/hear')

# 載入官方前處理函數
import importlib
audio_utils = importlib.import_module("hear.python.data_processing.audio_utils")
preprocess_audio = audio_utils.preprocess_audio

print("成功載入官方 preprocess_audio 函數！")

成功載入官方 preprocess_audio 函數！


In [10]:
def extract_hear_embeddings(chunks, model, preprocess_func, device):
    """
    使用官方 preprocess_audio 將音訊轉為 4D 張量，再送入 HeAR 模型提取 512-dim embeddings
    """
    if not chunks or len(chunks) == 0:
        return None

    embeddings = []
    model.eval()

    with torch.no_grad():
        for chunk in chunks:
            # 確保 chunk 形狀正確並加上 batch 維度: (1, 32000)
            input_array = np.expand_dims(chunk, axis=0)
            input_tensor = torch.Tensor(input_array)

            try:
                # 【關鍵修正】：使用官方前處理轉換成模型預期的 4D 頻譜圖張量
                processed_tensor = preprocess_func(input_tensor).to(device)

                # 執行推論 (官方範例指定使用 return_dict=True)
                output = model.forward(
                    processed_tensor,
                    return_dict=True,
                    output_hidden_states=True
                )

                # 提取 embedding (HeAR 的最終聲學特徵儲存在 pooler_output)
                if hasattr(output, 'pooler_output') and output.pooler_output is not None:
                    emb = output.pooler_output.cpu().detach().numpy().flatten()
                elif hasattr(output, 'last_hidden_state'):
                    emb = output.last_hidden_state.mean(dim=1).cpu().detach().numpy().flatten()
                else:
                    emb = output[0].mean(dim=1).cpu().detach().numpy().flatten()

                embeddings.append(emb)

            except Exception as e:
                print(f"Model Inference Error: {e}")
                continue

    # 對該音檔的所有 2 秒片段的特徵取平均
    return np.mean(embeddings, axis=0) if embeddings else None

In [11]:
# ================= 重新執行提取 =================
tqdm.pandas(desc="提取 Embeddings (官方前處理)")

embeddings_list = []
# 確保模型在正確的 device
model.to(device)

for idx, row in tqdm(df_clean.iterrows(), total=len(df_clean)):
    chunks = row['audio_chunks']
    try:
        # 將 preprocess_audio 作為參數傳入
        embedding = extract_hear_embeddings(chunks, model, preprocess_audio, device)
        embeddings_list.append(embedding)
    except Exception as e:
        print(f"Error on row {idx}: {e}")
        embeddings_list.append(None)

df_clean['embedding'] = embeddings_list
df_result = df_clean.dropna(subset=['embedding']).copy().reset_index(drop=True)

print(f"\n處理完成！有效的 row 數量: {len(df_result)}")
if len(df_result) > 0:
    print(f"第一個 embedding 的維度 (column 數量): {df_result['embedding'].iloc[0].shape}")

  1%|          | 4/455 [00:54<1:42:00, 13.57s/it]


KeyboardInterrupt: 

In [30]:
df_result.iloc[0]["embedding"].shape
print("="*50)
df_result
print(df_result.groupby("label").count())


       subject_id  file_path  audio_chunks  embedding
label                                                
0             324        324           324        324
1             131        131           131        131


### Test

In [19]:

# 1. Get the first data entry
first_row = df_clean.iloc[0]
first_chunks = first_row['audio_chunks']
file_info = first_row['file_path']

print(f"Testing with File: {file_info}")
print(f"Number of chunks to process: {len(first_chunks)}")

# 2. Perform extraction (using your final confirmed version)
try:
    # Ensure the model is on the correct device
    model.to(device)

    # Execute the extraction function
    test_embedding = extract_hear_embeddings(first_chunks, model, processor, device)

    # 3. Result verification
    if test_embedding is not None:
        print("\n--- Extraction Success ---")
        print(f"Embedding Shape: {test_embedding.shape}")
        print(f"Embedding Type: {type(test_embedding)}")

        # Check for anomalous values (NaN or Inf)
        if np.isnan(test_embedding).any():
            print("Warning: Embedding contains NaN values.")
        else:
            print("Data Quality: No NaNs detected.")

        # Display the first 5 value examples
        print(f"First 5 values: {test_embedding[:5]}")

    else:
        print("\n--- Extraction Failed ---")
        print("Reason: Function returned None (likely empty chunks or error in loop).")

except Exception as e:
    print(f"\n--- Unexpected Error ---")
    print(f"Error Message: {e}")

Testing with File: /content/drive/MyDrive/NUS_MSBA/DBA5102/Innovation_Challenge/dementia/Dan Ingram/daningram_15.wav
Number of chunks to process: 23

--- Extraction Success ---
Embedding Shape: (512,)
Embedding Type: <class 'numpy.ndarray'>
Data Quality: No NaNs detected.
First 5 values: [ 1.6047806 -8.284615   5.69111   15.247298   2.2522666]


## SVM Model

In [39]:
# 1. Prepare Data
# labels converted to y (1 for diagnosed, 0 for not diagnosed)
X = np.stack(df_result['embedding'].values)
y = df_result['label'].values

# 2. Split into final test set (Hold-out Test Set)
# Use stratify=y to ensure consistent positive/negative sample ratios in train/test sets
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# 3. Create Pipeline
# Order: Standardization -> PCA Dimensionality Reduction -> Linear SVM
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('svm', LinearSVC(class_weight='balanced', max_iter=10000, dual=False))
])

# 4. Define Hyperparameter Search Space
# We adjust both the number of PCA components and the SVM regularization parameter C
param_grid = {
    'pca__n_components': [10, 20, 30, 50, 0.95], # Number represents dimension, 0.95 represents retaining 95% variance
    'svm__C': [0.001, 0.01, 0.1, 1, 10]
}

# 5. Perform Cross-Validation and Search using Stratified K-Fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Considering imbalance, we use f1 score as the evaluation metric
grid_search = GridSearchCV(
    pipeline, param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=1
)

print("Starting Grid Search with Stratified 5-Fold CV...")
grid_search.fit(X_train_val, y_train_val)

# 6. Output Best Parameters and Results
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV F1-Score: {grid_search.best_score_:.4f}")

# 7. Evaluate on a completely unseen Test Set
y_pred = grid_search.predict(X_test)

print("\n--- Final Evaluation on Test Set ---")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Additionally calculate AUC-ROC (requires decision_function to get scores)
y_scores = grid_search.decision_function(X_test)
auc = roc_auc_score(y_test, y_scores)
print(f"Test Set AUC-ROC: {auc:.4f}")

Starting Grid Search with Stratified 5-Fold CV...
Fitting 5 folds for each of 25 candidates, totalling 125 fits

Best parameters: {'pca__n_components': 30, 'svm__C': 0.001}
Best CV F1-Score: 0.5059

--- Final Evaluation on Test Set ---
[[42 23]
 [10 16]]
              precision    recall  f1-score   support

           0       0.81      0.65      0.72        65
           1       0.41      0.62      0.49        26

    accuracy                           0.64        91
   macro avg       0.61      0.63      0.61        91
weighted avg       0.69      0.64      0.65        91

Test Set AUC-ROC: 0.6822


In [38]:
# SVC model with proper grouping
# ================= 1. Prepare Data =================
X = np.stack(df_result['embedding'].values)
y = df_result['label'].values
# Extract subject_id as group basis
groups = df_result['subject_id'].values

# ================= 2. Hold-out Split to Prevent Data Leakage =================
# Use GroupShuffleSplit to ensure that subjects in the training and test sets do not overlap
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_val, X_test = X[train_idx], X[test_idx]
y_train_val, y_test = y[train_idx], y[test_idx]
groups_train_val = groups[train_idx]
groups_test = groups[test_idx]

# ================= 3. Create Pipeline =================
# Replace LinearSVC with SVC(kernel='linear') and set probability=True
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('svm', SVC(kernel='linear', class_weight='balanced', probability=True, random_state=42))
])

# ================= 4. Define Hyperparameter Search Space =================
param_grid = {
    'pca__n_components': [10, 20, 30, 50, 0.95],
    'svm__C': [0.001, 0.01, 0.1, 1, 10]
}

# ================= 5. Execute GridSearchCV =================
# Use StratifiedGroupKFold to ensure no leakage during Cross-Validation
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline, param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=1
)

print("Starting Grid Search with Stratified Group 5-Fold CV...")
# When training, groups_train_val must be passed so that KFold knows how to split
grid_search.fit(X_train_val, y_train_val, groups=groups_train_val)

# ================= 6. Output Best Parameters and Results =================
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV F1-Score: {grid_search.best_score_:.4f}")

# ================= 7. Evaluate on a Separate Test Set =================
y_pred = grid_search.predict(X_test)

# Use predict_proba to get probabilities (the returned matrix has two columns, index 1 is the probability for class 1)
y_prob = grid_search.predict_proba(X_test)[:, 1]

print("\n--- Final Evaluation on Unseen Test Set ---")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Calculate AUC-ROC using the probability array
auc = roc_auc_score(y_test, y_prob)
print(f"Test Set AUC-ROC (using probability): {auc:.4f}")

# (Optional) Print the first few prediction probabilities for review
print("\n--- Sample Prediction Probabilities ---")
for i in range(min(5, len(y_test))):
    print(f"True Label: {y_test[i]} | Predicted: {y_pred[i]} | Probability (Dementia): {y_prob[i]:.4f}")

Starting Grid Search with Stratified Group 5-Fold CV...
Fitting 5 folds for each of 25 candidates, totalling 125 fits

Best parameters: {'pca__n_components': 30, 'svm__C': 10}
Best CV F1-Score: 0.5055

--- Final Evaluation on Unseen Test Set ---
Confusion Matrix:
[[47 28]
 [ 6 15]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.63      0.73        75
           1       0.35      0.71      0.47        21

    accuracy                           0.65        96
   macro avg       0.62      0.67      0.60        96
weighted avg       0.77      0.65      0.68        96

Test Set AUC-ROC (using probability): 0.7073

--- Sample Prediction Probabilities ---
True Label: 1 | Predicted: 1 | Probability (Dementia): 0.3610
True Label: 1 | Predicted: 1 | Probability (Dementia): 0.7761
True Label: 1 | Predicted: 1 | Probability (Dementia): 0.4465
True Label: 1 | Predicted: 1 | Probability (Dementia): 0.4428
True Label: 1 | Predicted: 0 | P

In [42]:
# Logistic regression model for more accurate probability display
# ================= 1. Prepare Data =================
X = np.stack(df_result['embedding'].values)
y = df_result['label'].values
# Extract subject_id as group basis
groups = df_result['subject_id'].values

# ================= 2. Hold-out Split to Prevent Data Leakage =================
# Use GroupShuffleSplit to ensure that subjects in the training and test sets do not overlap
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_val, X_test = X[train_idx], X[test_idx]
y_train_val, y_test = y[train_idx], y[test_idx]
groups_train_val = groups[train_idx]
groups_test = groups[test_idx]

# ================= 3. Create Pipeline =================
# Replace LinearSVC with SVC(kernel='linear') and set probability=True
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('clf', LogisticRegression(class_weight='balanced', random_state=42))
])

# ================= 4. Define Hyperparameter Search Space =================
param_grid = {
    'pca__n_components': [10, 20, 30, 50, 0.95],
    'clf__C': [0.001, 0.01, 0.1, 1, 10]
}

# ================= 5. Execute GridSearchCV =================
# Use StratifiedGroupKFold to ensure no leakage during Cross-Validation
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline, param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=1
)

print("Starting Grid Search with Stratified Group 5-Fold CV...")
# When training, groups_train_val must be passed so that KFold knows how to split
grid_search.fit(X_train_val, y_train_val, groups=groups_train_val)

# ================= 6. Output Best Parameters and Results =================
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV F1-Score: {grid_search.best_score_:.4f}")

# ================= 7. Evaluate on a Separate Test Set =================
y_pred = grid_search.predict(X_test)

# Use predict_proba to get probabilities (the returned matrix has two columns, index 1 is the probability for class 1)
y_prob = grid_search.predict_proba(X_test)[:, 1]

print("\n--- Final Evaluation on Unseen Test Set ---")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Calculate AUC-ROC using the probability array
auc = roc_auc_score(y_test, y_prob)
print(f"Test Set AUC-ROC (using probability): {auc:.4f}")

# (Optional) Print the first few prediction probabilities for review
print("\n--- Sample Prediction Probabilities ---")
for i in range(min(5, len(y_test))):
    print(f"True Label: {y_test[i]} | Predicted: {y_pred[i]} | Probability (Dementia): {y_prob[i]:.4f}")

Starting Grid Search with Stratified Group 5-Fold CV...
Fitting 5 folds for each of 25 candidates, totalling 125 fits

Best parameters: {'clf__C': 1, 'pca__n_components': 30}
Best CV F1-Score: 0.4603

--- Final Evaluation on Unseen Test Set ---
Confusion Matrix:
[[49 26]
 [ 6 15]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.65      0.75        75
           1       0.37      0.71      0.48        21

    accuracy                           0.67        96
   macro avg       0.63      0.68      0.62        96
weighted avg       0.78      0.67      0.69        96

Test Set AUC-ROC (using probability): 0.7041

--- Sample Prediction Probabilities ---
True Label: 1 | Predicted: 1 | Probability (Dementia): 0.5146
True Label: 1 | Predicted: 1 | Probability (Dementia): 0.9841
True Label: 1 | Predicted: 1 | Probability (Dementia): 0.7935
True Label: 1 | Predicted: 1 | Probability (Dementia): 0.7698
True Label: 1 | Predicted: 0 | Pr